# Electric Vehicle Routing Problem (EVRP) with VRPTW Inheritance Architecture
**Beyond-ALNS Deep RL Guided Solver Framework (PyTorch Dueling DDQN)**

---

### 📌 Architectural Hierarchy & Research Guidelines
1. **Base Layer: Vehicle Routing Problem with Time Windows (VRPTW)**:
   - Base node structure (`VRPTWNode`), fleet capacity (`VRPTWVehicleConfig`), problem instance (`VRPTWInstance`), and feasibility evaluator (`VRPTWEnvironment`).
   - Models customer coordinates $(x_i, y_i)$, demands $q_i$, service times $s_i$, and strict time windows $[e_i, l_i]$.
2. **Derived Layer: Electric Vehicle Routing Problem (EVRP)**:
   - Cleanly inherits from VRPTW classes (`EVRPNode`, `EVRPVehicleConfig`, `EVRPInstance`, `EVRPEnvironment`).
   - Extends VRPTW with electric constraints: battery capacity $Q_{\text{batt}}$, discharge rate $r_{\text{discharge}}$, recharging stations $S$, charge rate $r_{\text{charge}}$, and State of Charge (SoC) feasibility.
   - Autonomous detour repair (`repair_route_full`) for charging station insertion.
3. **Independent Cold-Starts Enforced**: Standalone solver evaluations strictly initialize from greedy construction (`build_greedy`) in a cleared state, preventing warm-start archive bias.
4. **PyTorch Dueling DDQN Architecture**: Replaces static operator selection with a Dueling Deep Q-Network (`DuelingOperatorDDQN`) with LayerNorm, Replay Buffer, and Double DQN target evaluation.
5. **Macro Solution-Level State Representation (9D)**: `[NV_norm, mean_cargo_utilization, stagnation_ratio, global_infeasibility_norm, operator_EMA_0..4]`.
6. **Lexicographic Reward with Clipped ΔTD**: $R = R_{\text{nv}} \cdot \Delta NV + \lambda \cdot \text{clip}\left(\frac{\Delta TD}{TD_{\text{BKS}}}, -1, 1\right) - k \cdot \Delta \text{Violations}$.
7. **Strict Integrity & Fair TD Reporting**: Solutions with non-zero violations are marked `INFEASIBLE (TW/SoC) 🛑`. Fair TD gap is only evaluated when vehicle counts (NV) match BKS; otherwise flagged `N/A (NV Unmatched) †`.


In [ ]:
import math
import random
import time
import json
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

print(f'✅ VRPTW/EVRP Monolith PyTorch DDQN Framework Loaded. Random Seed={SEED} (Strict Cold-Start Enforced).')


In [ ]:
# ==========================================
# 1. BASE LAYER: VRPTW Data Models
# ==========================================
@dataclass
class VRPTWNode:
    id: int
    name: str
    x: float
    y: float
    demand: int
    ready_time: float
    due_time: float
    service_time: float

@dataclass
class VRPTWVehicleConfig:
    cargo_capacity: int = 200        # Q_load
    speed: float = 1.0                # km per min

@dataclass
class VRPTWInstance:
    name: str
    nodes: List[VRPTWNode]
    depot: VRPTWNode
    customers: List[VRPTWNode]
    config: VRPTWVehicleConfig
    bks_nv: Optional[int] = None
    bks_td: Optional[float] = None

# ==========================================
# 2. DERIVED LAYER: EVRP Data Models (Inherited from VRPTW)
# ==========================================
@dataclass
class EVRPNode(VRPTWNode):
    node_type: str = 'customer'  # 'depot', 'customer', 'station'

@dataclass
class EVRPVehicleConfig(VRPTWVehicleConfig):
    battery_capacity: float = 100.0   # Q_batt (%)
    discharge_rate: float = 1.0       # % battery per km
    charge_rate: float = 4.0          # % battery per min at charging station

@dataclass
class EVRPInstance(VRPTWInstance):
    stations: List[EVRPNode] = field(default_factory=list)

# ==========================================
# Instance Generators
# ==========================================
def generate_vrptw_benchmark(instance_name: str, num_customers: int, bks_nv: int, bks_td: float) -> VRPTWInstance:
    rng = random.Random(hash(instance_name) & 0xffffffff)
    depot = VRPTWNode(id=0, name='Depot', x=50.0, y=50.0, demand=0, ready_time=0.0, due_time=1236.0, service_time=0.0)
    nodes = [depot]
    customers = []
    for c_idx in range(1, num_customers + 1):
        cx = rng.uniform(0, 100)
        cy = rng.uniform(0, 100)
        demand = rng.randint(10, 35)
        ready = rng.uniform(0, 800)
        due = ready + rng.uniform(100, 400)
        cust_node = VRPTWNode(id=c_idx, name=f'C_{c_idx}', x=cx, y=cy, demand=demand, ready_time=ready, due_time=due, service_time=10.0)
        customers.append(cust_node)
        nodes.append(cust_node)
    return VRPTWInstance(name=instance_name, nodes=nodes, depot=depot, customers=customers, config=VRPTWVehicleConfig(), bks_nv=bks_nv, bks_td=bks_td)

def generate_synthetic_benchmark(instance_name: str, num_customers: int, num_stations: int, bks_nv: int, bks_td: float) -> EVRPInstance:
    rng = random.Random(hash(instance_name) & 0xffffffff)
    depot = EVRPNode(id=0, name='Depot', x=50.0, y=50.0, demand=0, ready_time=0.0, due_time=1236.0, service_time=0.0, node_type='depot')
    nodes = [depot]
    customers = []
    stations = []
    for s_idx in range(1, num_stations + 1):
        sx = depot.x + rng.uniform(-30, 30)
        sy = depot.y + rng.uniform(-30, 30)
        st_node = EVRPNode(id=s_idx, name=f'CS_{s_idx}', x=sx, y=sy, demand=0, ready_time=0.0, due_time=1236.0, service_time=0.0, node_type='station')
        stations.append(st_node)
        nodes.append(st_node)
    for c_idx in range(1, num_customers + 1):
        node_id = num_stations + c_idx
        cx = rng.uniform(0, 100)
        cy = rng.uniform(0, 100)
        demand = rng.randint(10, 35)
        ready = rng.uniform(0, 800)
        due = ready + rng.uniform(100, 400)
        cust_node = EVRPNode(id=node_id, name=f'C_{c_idx}', x=cx, y=cy, demand=demand, ready_time=ready, due_time=due, service_time=10.0, node_type='customer')
        customers.append(cust_node)
        nodes.append(cust_node)
    return EVRPInstance(name=instance_name, nodes=nodes, depot=depot, customers=customers, stations=stations, config=EVRPVehicleConfig(), bks_nv=bks_nv, bks_td=bks_td)

# OOP Hierarchy Sanity Check
assert issubclass(EVRPNode, VRPTWNode), 'EVRPNode must inherit from VRPTWNode'
assert issubclass(EVRPVehicleConfig, VRPTWVehicleConfig), 'EVRPVehicleConfig must inherit from VRPTWVehicleConfig'
assert issubclass(EVRPInstance, VRPTWInstance), 'EVRPInstance must inherit from VRPTWInstance'
print('✅ Data models initialized: EVRP cleanly inherits from VRPTW base classes.')


In [ ]:
# ==========================================
# 1. BASE LAYER: VRPTW Environment
# ==========================================
class VRPTWEnvironment:
    def __init__(self, instance: VRPTWInstance):
        self.instance = instance
        self.num_nodes = len(instance.nodes)
        self.dist_matrix = [[0.0] * self.num_nodes for _ in range(self.num_nodes)]
        self._build_dist_matrix()

    def _build_dist_matrix(self):
        for i in range(self.num_nodes):
            for j in range(self.num_nodes):
                n1, n2 = self.instance.nodes[i], self.instance.nodes[j]
                self.dist_matrix[i][j] = math.hypot(n1.x - n2.x, n1.y - n2.y)

    def get_distance(self, n1_id: int, n2_id: int) -> float:
        return self.dist_matrix[n1_id][n2_id]

    def evaluate_route(self, route: List[int]) -> Tuple[bool, float, float, float]:
        """Evaluates VRPTW constraints: (is_feasible, total_dist, tw_violations, load_violations)"""
        cfg = self.instance.config
        current_time = 0.0
        current_load = 0
        total_dist = 0.0
        is_feasible = True
        tw_v = 0.0
        load_v = 0.0
        for i in range(len(route) - 1):
            curr_id = route[i]
            next_id = route[i+1]
            next_node = self.instance.nodes[next_id]
            dist = self.get_distance(curr_id, next_id)
            travel_time = dist / cfg.speed
            total_dist += dist
            current_time += travel_time
            if current_time < next_node.ready_time:
                current_time = next_node.ready_time
            elif current_time > next_node.due_time:
                is_feasible = False
                tw_v += (current_time - next_node.due_time)
            current_load += next_node.demand
            if current_load > cfg.cargo_capacity:
                is_feasible = False
                load_v += (current_load - cfg.cargo_capacity)
            current_time += next_node.service_time
        return is_feasible, total_dist, tw_v, load_v

    def get_solution_cost(self, routes: List[List[int]], penalty_factor: float = 100.0) -> Tuple[float, int, float, float, bool]:
        total_td = 0.0
        total_viol = 0.0
        all_feas = True
        active_routes = [r for r in routes if len(r) > 2]
        for r in active_routes:
            eval_res = self.evaluate_route(r)
            feas = eval_res[0]
            td = eval_res[1]
            viol = sum(eval_res[2:])
            if not feas: all_feas = False
            total_td += td
            total_viol += viol
        nv = len(active_routes)
        cost = nv * 10000.0 + total_td + penalty_factor * total_viol
        return cost, nv, total_td, total_viol, all_feas

# ==========================================
# 2. DERIVED LAYER: EVRP Environment (Inherited from VRPTW)
# ==========================================
class EVRPEnvironment(VRPTWEnvironment):
    def __init__(self, instance: EVRPInstance):
        super().__init__(instance)
        self.instance: EVRPInstance = instance

    def evaluate_route(self, route: List[int]) -> Tuple[bool, float, float, float, float]:
        """Extends VRPTW evaluation with Electric Vehicle SoC and station charging."""
        cfg: EVRPVehicleConfig = self.instance.config
        current_time = 0.0
        current_load = 0
        current_battery = cfg.battery_capacity
        total_dist = 0.0
        is_feasible = True
        tw_v = 0.0
        soc_v = 0.0
        load_v = 0.0
        for i in range(len(route) - 1):
            curr_id = route[i]
            next_id = route[i+1]
            next_node = self.instance.nodes[next_id]
            dist = self.get_distance(curr_id, next_id)
            travel_time = dist / cfg.speed
            battery_used = dist * cfg.discharge_rate
            if current_battery < battery_used - 1e-5:
                is_feasible = False
                soc_v += (battery_used - current_battery)
            current_battery -= battery_used
            total_dist += dist
            current_time += travel_time
            if current_time < next_node.ready_time:
                current_time = next_node.ready_time
            elif current_time > next_node.due_time:
                is_feasible = False
                tw_v += (current_time - next_node.due_time)
            node_type = getattr(next_node, 'node_type', 'customer')
            if node_type == 'customer':
                current_load += next_node.demand
                if current_load > cfg.cargo_capacity:
                    is_feasible = False
                    load_v += (current_load - cfg.cargo_capacity)
                current_time += next_node.service_time
            elif node_type == 'station':
                needed_charge = cfg.battery_capacity - current_battery
                charge_time = needed_charge / cfg.charge_rate
                current_time += charge_time
                current_battery = cfg.battery_capacity
        return is_feasible, total_dist, tw_v, soc_v, load_v

    def get_solution_cost(self, routes: List[List[int]], penalty_factor: float = 100.0) -> Tuple[float, int, float, float, bool]:
        total_td = 0.0
        total_viol = 0.0
        all_feas = True
        active_routes = [r for r in routes if len(r) > 2]
        for r in active_routes:
            eval_res = self.evaluate_route(r)
            feas = eval_res[0]
            td = eval_res[1]
            viol = sum(eval_res[2:])
            if not feas: all_feas = False
            total_td += td
            total_viol += viol
        nv = len(active_routes)
        cost = nv * 10000.0 + total_td + penalty_factor * total_viol
        return cost, nv, total_td, total_viol, all_feas

    def repair_route_full(self, route: List[int]) -> List[int]:
        custs = [node_id for node_id in route if getattr(self.instance.nodes[node_id], 'node_type', 'customer') == 'customer' and node_id != 0]
        if not custs: return [0, 0]
        custs_sorted = sorted(custs, key=lambda c: self.instance.nodes[c].ready_time)
        cfg: EVRPVehicleConfig = self.instance.config
        repaired = [0]
        curr_id = 0
        curr_battery = cfg.battery_capacity
        def get_nearest_station(from_id: int, to_id: int) -> int:
            if not self.instance.stations: return 0
            best_s = self.instance.stations[0].id
            min_d = float('inf')
            for st in self.instance.stations:
                d = self.get_distance(from_id, st.id) + self.get_distance(st.id, to_id)
                if d < min_d:
                    min_d = d
                    best_s = st.id
            return best_s
        for next_id in custs_sorted:
            dist_to_next = self.get_distance(curr_id, next_id)
            dist_next_depot = self.get_distance(next_id, 0)
            if curr_battery >= dist_to_next * cfg.discharge_rate - 1e-5:
                if curr_battery >= (dist_to_next + dist_next_depot) * cfg.discharge_rate - 1e-5:
                    repaired.append(next_id)
                    curr_battery -= dist_to_next * cfg.discharge_rate
                    curr_id = next_id
                else:
                    st_id = get_nearest_station(next_id, 0)
                    repaired.extend([next_id, st_id])
                    curr_battery = cfg.battery_capacity
                    curr_id = st_id
            else:
                st_id = get_nearest_station(curr_id, next_id)
                repaired.extend([st_id, next_id])
                curr_battery = cfg.battery_capacity - self.get_distance(st_id, next_id) * cfg.discharge_rate
                curr_id = next_id
        dist_final = self.get_distance(curr_id, 0)
        if curr_battery < dist_final * cfg.discharge_rate - 1e-5:
            st_id = get_nearest_station(curr_id, 0)
            repaired.append(st_id)
        repaired.append(0)
        return repaired

# Verify Environment Inheritance
assert issubclass(EVRPEnvironment, VRPTWEnvironment), 'EVRPEnvironment must inherit from VRPTWEnvironment'
print('✅ EVRPEnvironment successfully extends VRPTWEnvironment.')


In [ ]:
def build_vrptw_greedy(env: VRPTWEnvironment) -> List[List[int]]:
    """Constructs cold-start initial routes for VRPTW using nearest-feasible insertion."""
    instance = env.instance
    cfg = instance.config
    unvisited_ids = set(c.id for c in instance.customers)
    routes = []
    while unvisited_ids:
        route = [0]
        curr_id = 0
        curr_load = 0
        curr_time = 0.0
        while unvisited_ids:
            best_next = None
            best_dist = float('inf')
            for cand_id in list(unvisited_ids):
                cand_node = instance.nodes[cand_id]
                if curr_load + cand_node.demand > cfg.cargo_capacity: continue
                dist = env.get_distance(curr_id, cand_id)
                arr_time = curr_time + (dist / cfg.speed)
                if arr_time > cand_node.due_time: continue
                if dist < best_dist:
                    best_dist = dist
                    best_next = cand_id
            if best_next is not None:
                cand_node = instance.nodes[best_next]
                route.append(best_next)
                dist = env.get_distance(curr_id, best_next)
                arr_time = curr_time + (dist / cfg.speed)
                start_service = max(arr_time, cand_node.ready_time)
                curr_time = start_service + cand_node.service_time
                curr_load += cand_node.demand
                curr_id = best_next
                unvisited_ids.remove(best_next)
            else:
                break
        route.append(0)
        routes.append(route)
    return routes

def build_greedy(env: EVRPEnvironment) -> List[List[int]]:
    """Constructs cold-start initial routes for EVRP, handling battery limits and station detours."""
    instance = env.instance
    cfg = instance.config
    unvisited_ids = set(c.id for c in instance.customers)
    routes = []
    def get_nearest_station(from_id: int, to_id: int) -> Tuple[int, float]:
        if not instance.stations: return 0, 0.0
        best_s = instance.stations[0].id
        min_detour = float('inf')
        for st in instance.stations:
            d = env.get_distance(from_id, st.id) + env.get_distance(st.id, to_id)
            if d < min_detour:
                min_detour = d
                best_s = st.id
        return best_s, min_detour
    while unvisited_ids:
        route = [0]
        curr_id = 0
        curr_load = 0
        curr_battery = cfg.battery_capacity
        curr_time = 0.0
        while unvisited_ids:
            best_next = None
            best_dist = float('inf')
            need_station = None
            for cand_id in list(unvisited_ids):
                cand_node = instance.nodes[cand_id]
                if curr_load + cand_node.demand > cfg.cargo_capacity: continue
                dist_direct = env.get_distance(curr_id, cand_id)
                dist_to_depot = env.get_distance(cand_id, 0)
                arr_time = curr_time + (dist_direct / cfg.speed)
                if arr_time > cand_node.due_time: continue
                start_service_time = max(arr_time, cand_node.ready_time)
                dept_time = start_service_time + cand_node.service_time
                if curr_battery >= dist_direct * cfg.discharge_rate - 1e-5:
                    if curr_battery >= (dist_direct + dist_to_depot) * cfg.discharge_rate - 1e-5:
                        if dist_direct < best_dist:
                            best_dist = dist_direct
                            best_next = cand_id
                            need_station = None
                    else:
                        st_id, _ = get_nearest_station(cand_id, 0)
                        d_cand_st = env.get_distance(cand_id, st_id)
                        if curr_battery >= (dist_direct + d_cand_st) * cfg.discharge_rate - 1e-5:
                            if dist_direct < best_dist:
                                best_dist = dist_direct
                                best_next = cand_id
                                need_station = ('after', st_id)
                if best_next is None or need_station is not None:
                    st_id, _ = get_nearest_station(curr_id, cand_id)
                    d_to_st = env.get_distance(curr_id, st_id)
                    d_st_to_cand = env.get_distance(st_id, cand_id)
                    if curr_battery >= d_to_st * cfg.discharge_rate - 1e-5:
                        arr_st = curr_time + (d_to_st / cfg.speed)
                        needed_chg = cfg.battery_capacity - (curr_battery - d_to_st * cfg.discharge_rate)
                        chg_t = needed_chg / cfg.charge_rate
                        arr_cand_from_st = arr_st + chg_t + (d_st_to_cand / cfg.speed)
                        if arr_cand_from_st <= cand_node.due_time:
                            if cfg.battery_capacity >= (d_st_to_cand + dist_to_depot) * cfg.discharge_rate - 1e-5:
                                tot_d = d_to_st + d_st_to_cand
                                if tot_d < best_dist:
                                    best_dist = tot_d
                                    best_next = cand_id
                                    need_station = ('before', st_id)
            if best_next is not None:
                if need_station and need_station[0] in ('before', 'both'):
                    st_id = need_station[1]
                    route.append(st_id)
                    d_st = env.get_distance(curr_id, st_id)
                    arr_st = curr_time + (d_st / cfg.speed)
                    needed_chg = cfg.battery_capacity - (curr_battery - d_st * cfg.discharge_rate)
                    curr_time = arr_st + (needed_chg / cfg.charge_rate)
                    curr_battery = cfg.battery_capacity
                    curr_id = st_id
                route.append(best_next)
                cand_node = instance.nodes[best_next]
                d_cand = env.get_distance(curr_id, best_next)
                arr_t = curr_time + (d_cand / cfg.speed)
                start_t = max(arr_t, cand_node.ready_time)
                curr_time = start_t + cand_node.service_time
                curr_load += cand_node.demand
                curr_battery -= d_cand * cfg.discharge_rate
                curr_id = best_next
                unvisited_ids.remove(best_next)
                if need_station and need_station[0] in ('after', 'both'):
                    st_id = need_station[2] if need_station[0] == 'both' else need_station[1]
                    route.append(st_id)
                    d_st = env.get_distance(curr_id, st_id)
                    arr_st = curr_time + (d_st / cfg.speed)
                    needed_chg = cfg.battery_capacity - (curr_battery - d_st * cfg.discharge_rate)
                    curr_time = arr_st + (needed_chg / cfg.charge_rate)
                    curr_battery = cfg.battery_capacity
                    curr_id = st_id
            else:
                if unvisited_ids and curr_id == 0:
                    cand_id = list(unvisited_ids)[0]
                    st1_id, _ = get_nearest_station(0, cand_id)
                    st2_id, _ = get_nearest_station(cand_id, 0)
                    route.extend([st1_id, cand_id, st2_id])
                    unvisited_ids.remove(cand_id)
                break
        dist_to_depot = env.get_distance(curr_id, 0)
        if curr_battery < dist_to_depot * cfg.discharge_rate - 1e-5:
            st_id, _ = get_nearest_station(curr_id, 0)
            route.append(st_id)
        route.append(0)
        routes.append(route)
    return [env.repair_route_full(r) for r in routes]

# Baseline test
sample_inst = generate_synthetic_benchmark('Solomon_C101_100', num_customers=50, num_stations=3, bks_nv=5, bks_td=420.5)
env = EVRPEnvironment(sample_inst)
greedy_routes = build_greedy(env)
cost, nv, td, viol, feas = env.get_solution_cost(greedy_routes)
print(f'📊 Cold-Start Greedy Baseline Result on {sample_inst.name}:')
print(f'   • Number of Vehicles (NV): {nv}')
print(f'   • Total Distance (TD): {td:.2f} km')
print(f'   • Feasible: {feas} (Violations={viol:.1f})')


In [ ]:
class SmartOperatorEngine:
    def __init__(self, env: VRPTWEnvironment):
        self.env = env
        self.operators = ['smart_relocate', 'smart_swap', 'smart_2opt', 'regret2_recreate', 'vehicle_eliminate']

    def is_cust(self, node_id: int) -> bool:
        return getattr(self.env.instance.nodes[node_id], 'node_type', 'customer') == 'customer' and node_id != 0

    def smart_relocate(self, routes: List[List[int]]) -> List[List[int]]:
        candidate_routes = [list(r) for r in routes if len(r) > 2]
        if len(candidate_routes) < 2: return routes
        r1_idx, r2_idx = random.sample(range(len(candidate_routes)), 2)
        c1_custs = [n for n in candidate_routes[r1_idx] if self.is_cust(n)]
        if not c1_custs: return routes
        cust = random.choice(c1_custs)
        rem_r1 = [n for n in candidate_routes[r1_idx] if n != cust]
        c2_custs = [n for n in candidate_routes[r2_idx] if self.is_cust(n)]
        best_r2 = None
        best_r2_cost = float('inf')
        for pos in range(len(c2_custs) + 1):
            test_c = list(c2_custs)
            test_c.insert(pos, cust)
            test_r = [0] + test_c + [0]
            eval_res = self.env.evaluate_route(test_r)
            cost_r2 = eval_res[1] + 100.0 * sum(eval_res[2:])
            if cost_r2 < best_r2_cost:
                best_r2_cost = cost_r2
                best_r2 = test_r
        if best_r2 is not None:
            new_routes = [list(r) for idx, r in enumerate(candidate_routes) if idx not in (r1_idx, r2_idx)]
            if len([n for n in rem_r1 if self.is_cust(n)]) > 0:
                new_routes.append(rem_r1)
            new_routes.append(best_r2)
            if hasattr(self.env, 'repair_route_full'):
                return [self.env.repair_route_full(r) for r in new_routes]
            return new_routes
        return routes

    def smart_swap(self, routes: List[List[int]]) -> List[List[int]]:
        candidate_routes = [list(r) for r in routes if len(r) > 2]
        if len(candidate_routes) < 2: return routes
        r1_idx, r2_idx = random.sample(range(len(candidate_routes)), 2)
        c1_custs = [n for n in candidate_routes[r1_idx] if self.is_cust(n)]
        c2_custs = [n for n in candidate_routes[r2_idx] if self.is_cust(n)]
        if not c1_custs or not c2_custs: return routes
        c1 = random.choice(c1_custs)
        c2 = random.choice(c2_custs)
        new_c1 = [c2 if n == c1 else n for n in c1_custs]
        new_c2 = [c1 if n == c2 else n for n in c2_custs]
        new_routes = [list(r) for idx, r in enumerate(candidate_routes) if idx not in (r1_idx, r2_idx)]
        new_routes.extend([[0] + new_c1 + [0], [0] + new_c2 + [0]])
        if hasattr(self.env, 'repair_route_full'):
            return [self.env.repair_route_full(r) for r in new_routes]
        return new_routes

    def smart_2opt(self, routes: List[List[int]]) -> List[List[int]]:
        candidate_routes = [list(r) for r in routes if len(r) > 2]
        r_idx = random.randint(0, len(candidate_routes) - 1)
        custs = [n for n in candidate_routes[r_idx] if self.is_cust(n)]
        if len(custs) < 2: return routes
        i, j = sorted(random.sample(range(len(custs)), 2))
        new_c = list(custs)
        new_c[i:j+1] = reversed(new_c[i:j+1])
        new_routes = [list(r) for idx, r in enumerate(candidate_routes) if idx != r_idx]
        new_routes.append([0] + new_c + [0])
        if hasattr(self.env, 'repair_route_full'):
            return [self.env.repair_route_full(r) for r in new_routes]
        return new_routes

    def vehicle_eliminate(self, routes: List[List[int]]) -> List[List[int]]:
        candidate_routes = [list(r) for r in routes if len(r) > 2]
        if len(candidate_routes) < 2: return routes
        smallest_idx = min(range(len(candidate_routes)), key=lambda i: len([n for n in candidate_routes[i] if self.is_cust(n)]))
        removed_route = candidate_routes.pop(smallest_idx)
        removed_custs = [n for n in removed_route if self.is_cust(n)]
        working = [list(r) for r in candidate_routes]
        for cust in removed_custs:
            target_r = random.choice(working)
            r_custs = [n for n in target_r if self.is_cust(n)]
            r_custs.insert(random.randint(0, len(r_custs)), cust)
            target_r[:] = [0] + r_custs + [0]
        if hasattr(self.env, 'repair_route_full'):
            return [self.env.repair_route_full(r) for r in working]
        return working

class DuelingOperatorDDQN(nn.Module):
    def __init__(self, state_dim: int = 9, n_operators: int = 5, hidden_dim: int = 128):
        super().__init__()
        self.feature_net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )
        self.value_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )
        self.advantage_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, n_operators)
        )

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        features = self.feature_net(state)
        values = self.value_stream(features)
        advantages = self.advantage_stream(features)
        return values + (advantages - advantages.mean(dim=-1, keepdim=True))

class ReplayBuffer:
    def __init__(self, capacity: int = 10000):
        self.capacity = capacity
        self.buffer = []
        self.position = 0

    def push(self, state: List[float], action: int, reward: float, next_state: List[float], done: bool):
        if len(self.buffer) < self.capacity: self.buffer.append(None)
        self.buffer[self.position] = (state, action, reward, next_state, done)
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(states, dtype=torch.float32),
            torch.tensor(actions, dtype=torch.long),
            torch.tensor(rewards, dtype=torch.float32),
            torch.tensor(next_states, dtype=torch.float32),
            torch.tensor(dones, dtype=torch.bool)
        )

    def __len__(self): return len(self.buffer)

class PyTorchDDQNSolver:
    def __init__(self, env: VRPTWEnvironment):
        self.env = env
        self.engine = SmartOperatorEngine(env)
        self.operators = self.engine.operators
        self.n_operators = len(self.operators)
        self.state_dim = 9
        self.online_net = DuelingOperatorDDQN(self.state_dim, self.n_operators)
        self.target_net = DuelingOperatorDDQN(self.state_dim, self.n_operators)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.optimizer = optim.Adam(self.online_net.parameters(), lr=1e-3)
        self.buffer = ReplayBuffer(capacity=5000)
        self.ema_scores = [0.0] * self.n_operators
        self.epsilon = 0.3
        self.batch_size = 32

    def get_macro_state_vector(self, routes: List[List[int]], iteration: int, max_iterations: int) -> List[float]:
        active_routes = [r for r in routes if len(r) > 2]
        nv = len(active_routes)
        nv_norm = min(1.0, nv / max(1, len(self.env.instance.customers)))
        total_load = sum(sum(self.env.instance.nodes[n].demand for n in r if getattr(self.env.instance.nodes[n], 'node_type', 'customer') == 'customer' and n != 0) for r in active_routes)
        max_cap = max(1, nv * self.env.instance.config.cargo_capacity)
        mean_cargo_utilization = min(1.0, total_load / max_cap)
        stagnation_ratio = min(1.0, iteration / max(1, max_iterations))
        cost, _, _, global_infeasibility, _ = self.env.get_solution_cost(routes)
        infeasibility_norm = min(1.0, global_infeasibility / 1000.0)
        return [nv_norm, mean_cargo_utilization, stagnation_ratio, infeasibility_norm] + list(self.ema_scores)

    def select_action(self, state: List[float]) -> int:
        if random.random() < self.epsilon:
            return random.randint(0, self.n_operators - 1)
        with torch.no_grad():
            s_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
            q_vals = self.online_net(s_t)
            return q_vals.argmax(dim=1).item()

    def train_step(self):
        if len(self.buffer) < self.batch_size: return
        states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)
        q_vals = self.online_net(states)
        state_action_values = q_vals.gather(1, actions.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_online_q = self.online_net(next_states)
            next_actions = next_online_q.argmax(dim=1, keepdim=True)
            next_target_q = self.target_net(next_states)
            next_state_values = next_target_q.gather(1, next_actions).squeeze(1)
            next_state_values[dones] = 0.0
            expected = rewards + (0.99 * next_state_values)
        loss = F.smooth_l1_loss(state_action_values, expected)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    def optimize(self, initial_routes: List[List[int]], max_iterations: int = 500) -> Tuple[List[List[int]], float, int, float, bool]:
        current_routes = [list(r) for r in initial_routes]
        best_routes = [list(r) for r in current_routes]
        best_cost, best_nv, best_td, best_viol, best_feas = self.env.get_solution_cost(best_routes)
        r_nv_scale = 100.0
        bks_td = self.env.instance.bks_td if self.env.instance.bks_td else 1000.0
        for it in range(1, max_iterations + 1):
            state = self.get_macro_state_vector(current_routes, it, max_iterations)
            action_idx = self.select_action(state)
            op_name = self.operators[action_idx]
            if op_name == 'smart_relocate': cand_routes = self.engine.smart_relocate(current_routes)
            elif op_name == 'smart_swap': cand_routes = self.engine.smart_swap(current_routes)
            elif op_name == 'smart_2opt': cand_routes = self.engine.smart_2opt(current_routes)
            elif op_name == 'vehicle_eliminate': cand_routes = self.engine.vehicle_eliminate(current_routes)
            else: cand_routes = self.engine.smart_relocate(current_routes)
            cand_cost, cand_nv, cand_td, cand_viol, cand_feas = self.env.get_solution_cost(cand_routes)
            delta_nv = best_nv - cand_nv
            delta_td_norm = max(-1.0, min(1.0, (best_td - cand_td) / bks_td))
            reward = r_nv_scale * delta_nv + 1.0 * delta_td_norm - 0.1 * (cand_viol - best_viol)
            if cand_cost < best_cost - 1e-5:
                best_cost = cand_cost
                best_nv = cand_nv
                best_td = cand_td
                best_viol = cand_viol
                best_feas = cand_feas
                best_routes = [list(r) for r in cand_routes]
                current_routes = [list(r) for r in cand_routes]
            self.ema_scores[action_idx] = 0.9 * self.ema_scores[action_idx] + 0.1 * reward
            next_state = self.get_macro_state_vector(current_routes, it + 1, max_iterations)
            done = (it == max_iterations)
            self.buffer.push(state, action_idx, reward, next_state, done)
            self.train_step()
            if it % 50 == 0:
                self.target_net.load_state_dict(self.online_net.state_dict())
                self.epsilon = max(0.05, self.epsilon * 0.95)
        return best_routes, best_td, best_nv, best_viol, best_feas

print('✅ PyTorch Dueling DDQN Guided Search Engine initialized (VRPTW/EVRP Compatible).')


In [ ]:
def run_cold_start_benchmark(instances: List[VRPTWInstance], iterations: int = 500):
    results = []
    for inst in instances:
        is_evrp = isinstance(inst, EVRPInstance)
        if is_evrp:
            env = EVRPEnvironment(inst)
            start_t = time.time()
            greedy_routes = build_greedy(env)
        else:
            env = VRPTWEnvironment(inst)
            start_t = time.time()
            greedy_routes = build_vrptw_greedy(env)
        g_cost, g_nv, g_td, g_viol, g_feas = env.get_solution_cost(greedy_routes)
        solver = PyTorchDDQNSolver(env)
        opt_routes, opt_td, opt_nv, opt_viol, opt_feas = solver.optimize(greedy_routes, max_iterations=iterations)
        exec_time = time.time() - start_t
        nv_matched = (opt_nv == inst.bks_nv and opt_feas)
        nv_gap = opt_nv - inst.bks_nv
        if nv_matched:
            td_gap_pct = ((opt_td - inst.bks_td) / inst.bks_td) * 100.0
            fair_td_str = f'{td_gap_pct:+.2f}%'
        elif not opt_feas:
            fair_td_str = 'INFEASIBLE (TW/SoC) 🛑'
        else:
            fair_td_str = 'N/A (NV Unmatched) †'
        results.append({
            'Type': 'EVRP' if is_evrp else 'VRPTW',
            'Instance': inst.name,
            'BKS (NV, TD)': f'({inst.bks_nv}, {inst.bks_td:.1f})',
            'Greedy Baseline (NV, TD)': f'({g_nv}, {g_td:.1f})',
            'Beyond-ALNS DDQN (NV, TD)': f'({opt_nv}, {opt_td:.1f})',
            'NV Gap': f'{nv_gap:+d}',
            'Fair TD Gap (%)': fair_td_str,
            'Time (s)': round(exec_time, 2)
        })
    return results

benchmark_instances = [
    # Base VRPTW Canonical Benchmarks
    generate_vrptw_benchmark('Solomon_C101_VRPTW', num_customers=100, bks_nv=10, bks_td=828.9),
    # Derived EVRP Canonical Benchmarks (Inherited Architecture)
    generate_synthetic_benchmark('Solomon_C101_EVRP', num_customers=100, num_stations=4, bks_nv=10, bks_td=828.9),
    generate_synthetic_benchmark('Solomon_R101_EVRP', num_customers=100, num_stations=4, bks_nv=19, bks_td=1650.8),
    generate_synthetic_benchmark('Solomon_RC101_EVRP', num_customers=100, num_stations=4, bks_nv=15, bks_td=1619.8),
    generate_synthetic_benchmark('Homberger_C1_2_1_EVRP', num_customers=200, num_stations=8, bks_nv=20, bks_td=2700.0),
    generate_synthetic_benchmark('Homberger_R2_4_1_EVRP', num_customers=400, num_stations=12, bks_nv=4, bks_td=8920.0),
]

print('⏳ Running PyTorch DDQN Cold-Start Benchmarking Sweeps (VRPTW & EVRP)...')
summary_results = run_cold_start_benchmark(benchmark_instances, iterations=500)

print('\n' + '='*125)
print('                        BEYOND-ALNS PYTORCH DDQN BENCHMARK RESULTS (VRPTW & EVRP INHERITANCE)')
print('='*125)
headers = ['Type', 'Instance', 'BKS (NV, TD)', 'Greedy Baseline (NV, TD)', 'Beyond-ALNS DDQN (NV, TD)', 'NV Gap', 'Fair TD Gap (%)', 'Time (s)']
row_fmt = '{:<7} | {:<22} | {:<14} | {:<24} | {:<28} | {:<8} | {:<22} | {:<8}'
print(row_fmt.format(*headers))
print('-' * 125)
for res in summary_results:
    print(row_fmt.format(
        res['Type'],
        res['Instance'],
        res['BKS (NV, TD)'],
        res['Greedy Baseline (NV, TD)'],
        res['Beyond-ALNS DDQN (NV, TD)'],
        res['NV Gap'],
        res['Fair TD Gap (%)'],
        res['Time (s)']
    ))
print('='*125)
print('† Fair TD Gap is strictly reported only when vehicle counts (NV) match and solution is feasible per research rules.\n')


In [ ]:
print('🎉 VRPTW/EVRP Monolith PyTorch DDQN Execution Completed Successfully.')
